# Stage 20B — Reference-3-mer-conditioned HAR motif sensitivity analysis

**Goal:** independent context-conditioned reanalysis of the completed Stage 20A results: 152 **provisional** human–chimp HAR ortholog pairs × six JASPAR2024 TF motifs. Uses exactly the same Stage 20A statistic and BH correction across **152 HAR families**, but replaces the reference-base-only substitution-placement null with a **chimp-reference trinucleotide (3-mer) context–matched** substitution-placement null.

**Status:** This is a *computational sensitivity test*, **not** an empirically validated human-lineage gain of TF binding. The chimp reference is a comparator sequence, not proven ancestral state. It does **not** preserve the final sequence's dinucleotide frequencies, provide matched non-HAR genomic negative controls, establish TF occupancy, or resolve original-author HAR interval provenance. TF motif relative-score cutoff ≥0.90 is exploratory, not a FIMO-calibrated p-value.

**Why faster:** Stage 20A showed 136 of 152 HARs have a zero-valued nonnegative maximum PWM-difference statistic. These receive mathematically exact p=1; only the 16 nonzero-statistic HARs need permutation simulation. The familywise BH comparison still includes **all 152 HARs**.

**How to run:** `Runtime → Run all` on Google Colab. There is no Google Drive mount or file upload. All previous input sequences, 152-HAR alignment metadata and Stage 20A result CSVs are embedded and SHA256-checked. The six motif matrices are read from the same checksum-pinned public JASPAR2024 MEME source as Stage 20A (internet required). Default **4,999 permutations per nonzero HAR**. Results checkpoint after *each HAR*; if Colab interrupts, run the notebook again in the same runtime to resume. The last cell triggers browser download **from the notebook process**, avoiding the previous subprocess/kernel error.

In [ ]:
# Step 1 — Restore and verify the embedded original research files
from pathlib import Path
import hashlib,base64,io,zipfile,json
ROOT=Path('/content') if Path('/content').exists() else Path.cwd()
PAYLOAD="""..."""
EXPECTED_HASHES = {'STAGE20A_152_PROVISIONAL_HUMAN.fa': 'b1d391c1bcfed507d6a83d36383f13e848a3a0ccb45937f1b77ddffd496db9eb', 'STAGE20A_152_PROVISIONAL_CHIMP.fa': 'ceecb00ddeda21a5458ff0bf25fbaf0b4779d6511921d9c07bb1d259b5fffc1c', 'STAGE20A_152_ALIGNMENT_QC.csv': 'ea4ca8e7a17cb042e41ce20d803f9d8f871da9efb0547414eb9d396d4a1f483e', 'STAGE20A_S1_HAR_152.csv': '7af7bacb23c96f0492d8af004cab6d14453013ae7fbfae3115c80c8db2472ac3', 'STAGE20A_S2_912_TF.csv': '6ee2305db1404ba71fbee3ca417d3fb2cb0abaafb5a2e3a2f2be2718f2806b01', 'HAR_Stage20A_ConditionalNull_NoDrive.py': '91fd8f3d75c94ceb590aeadde48fa8a279768ff7046fceeac7f1e5a6bea62920', 'HAR_Stage20A_Fast_Resumable.py': '987938476377ad75325fdd22ca534383f37e121fcf58fbf51311dc18ca71874e', 'HAR_Stage20B_3mer_ContextNull_NoDrive.py': '1c76c9cc793b4477e89b779bc98f2f8e032fb3d1c2f97ec7b9604e8e03017b50'}
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as z:
    assert set(z.namelist())==set(EXPECTED_HASHES), 'Unexpected embedded file set'
    for name in z.namelist():
        raw=z.read(name)
        digest=hashlib.sha256(raw).hexdigest()
        assert digest==EXPECTED_HASHES[name], 'Corrupt embedded payload '+name
        target=ROOT/name
        if target.exists() and hashlib.sha256(target.read_bytes()).hexdigest()==digest:
            print('Existing input OK:',name)
        else:
            target.write_bytes(raw)
            print('Restored:',name)
print('CHECKED:',len(EXPECTED_HASHES),'complete pinned research inputs')

In [ ]:
# Step 2 — Install dependencies only if missing
import importlib.util,subprocess,sys
for mod,req in [('numpy','numpy>=1.24'),('Bio','biopython>=1.84'),('numba','numba>=0.59'),('requests','requests>=2.31')]:
    if importlib.util.find_spec(mod) is None:
        print('Installing',req)
        subprocess.check_call([sys.executable,'-m','pip','install','-q',req])
import numba,numpy,Bio,requests
print('Dependencies ready:', 'numpy',numpy.__version__,'numba',numba.__version__)

In [ ]:
# Step 3 — Run Stage 20B in the notebook kernel (NOT a subprocess)
# 4,999 simulations per nonzero HAR; override via os.environ['HAR20B_PERMS'] if needed.
import sys,os,traceback,zipfile,importlib
from pathlib import Path
sys.path.insert(0,str(ROOT))
os.environ.setdefault('HAR20B_PERMS','4999')
try:
    import HAR_Stage20B_3mer_ContextNull_NoDrive as research
    research=importlib.reload(research)
    result = research.main()
    print('SUCCESS: Stage20B research ZIP created:',ROOT/'HAR_Stage20B_ContextMatched_Results.zip')
except Exception:
    traceback.print_exc()
    folder=ROOT/'HAR_Stage20B_3mer_ContextNull_Results'
    diagnostic_zip=ROOT/'HAR_Stage20B_Diagnostics_and_Partial.zip'
    with zipfile.ZipFile(diagnostic_zip,'w',zipfile.ZIP_DEFLATED) as z:
        if folder.exists():
            for p in folder.iterdir():
                if p.is_file():z.write(p,arcname=p.name)
    print('ERROR: Partial results preserved at',diagnostic_zip)
    raise

In [ ]:
# Step 4 — Download from actual Colab notebook kernel, never inside a subprocess
from pathlib import Path
path=ROOT/'HAR_Stage20B_ContextMatched_Results.zip'
if path.is_file():
    print('Complete output:',path,'size:',path.stat().st_size,'bytes')
    try:
        from google.colab import files
        files.download(str(path))
    except ImportError:
        print('Non-Colab environment: retrieve ZIP from printed path')
else:
    print('No complete ZIP. Use the partial checkpoint from the next cell.')

In [ ]:
# Optional recovery — use only if the run failed or the browser download was blocked.
from pathlib import Path
import zipfile
folder=ROOT/'HAR_Stage20B_3mer_ContextNull_Results'
backup=ROOT/'HAR_Stage20B_Diagnostics_and_Partial.zip'
with zipfile.ZipFile(backup,'w',zipfile.ZIP_DEFLATED) as z:
    if folder.exists():
        for p in folder.iterdir():
            if p.is_file():z.write(p,arcname=p.name)
print('Partial checkpoint:',backup,backup.stat().st_size,'bytes')
try:
    from google.colab import files
    files.download(str(backup))
except ImportError:pass